import config

In [1]:

import sys
import os
import tifffile
import numpy as np
import cupy as cp
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt

sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)
from utils.IO import readND2Frame, readMeta
from utils.simulation import generateMotion_Biophysical
from utils import calFlow3d_Wei_v1, calFlowCrossResolution
from utils import visualization as vis
from utils import preprocess as prep
from utils import option


root_dir = Path('/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos')
# root_dir = Path('/home/cyf/wbi/Virginia/code/src/wholistic_registration/exp/simulation/repos')
zslices = [2, 10, 18]  # 选择 z slice

experiment_groups = ['art_R', 'amp', 'noise_level']  # 对应三组实验
results = {}  # 保存 MSE
results = {group: {} for group in experiment_groups}
repo_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith('rep')])
num_repos = len(repo_dirs)
print(f"Found {num_repos} repos: {[d.name for d in repo_dirs]}")

CuPy is available with CUDA - using GPU acceleration
Found 10 repos: ['rep01', 'rep02', 'rep03', 'rep04', 'rep05', 'rep06', 'rep07', 'rep08', 'rep09', 'rep10']


raw_wbi & map_wbi

In [ ]:
from importlib import reload
import time
from utils import visualization
from utils import simulation
reload(simulation)
reload(calFlow3d_Wei_v1)
import cupy as cp
cp.cuda.Device(1).use()
for repo_idx, repo_dir in enumerate(repo_dirs):
    print(f"\n=== Processing Repo {repo_dir.name} ({repo_idx+1}/{num_repos}) ===")

    for group in experiment_groups:
        group_dir = repo_dir / group
        if not group_dir.exists():
            print(f"Warning: {group_dir} not exist, skipping")
            continue
        for subfolder in sorted(group_dir.iterdir(), key=lambda x: int(x.name)):
            if not subfolder.is_dir():
                continue
            exp_level = subfolder.name
            print(f"  Processing {group}/{exp_level}")
            ref_path = subfolder / 'ref.tif'
            mov_path = subfolder / 'mov.tif'
            motion_path = subfolder / 'motion.tif'
            if not ref_path.exists() or not mov_path.exists():
                print(f"    Missing file, skipping")
                continue
            ref_data = tifffile.imread(ref_path)[..., 2:19]
            move_data = tifffile.imread(mov_path)[..., zslices]
            full_motion=tifffile.imread(motion_path)
            motion_data = tifffile.imread(motion_path)[:,:,zslices,:]
            # print(np.mean(motion_data[:,:,:,2]**2))
            # print(np.mean(motion_data[:,:,:,0]**2))
            # print(np.mean(motion_data[:,:,:,1]**2))
            ref_data_clip = ref_data[:, :, [0,8,16]]
            option['zRatio'] = 3.0769230769230766 * 8
            option['zRatio_HR'] = 3.0769230769230766
            option['layer'] = 3
            option['iter'] = 10
            option['r']=5
            option['smoothPenalty_raw'] = 0.03
            option['mask_ref'] = np.zeros(ref_data.shape, dtype=np.bool_)
            option['mask_mov'] = np.zeros(move_data.shape, dtype=np.bool_)
            Pnltfactor = prep.getSmPnltNormFctr(ref_data, option)
            option['smoothPenalty'] = Pnltfactor * option['smoothPenalty_raw']
            option['movRange'] = 10
            option['wrong_region_enable']=False
            # try:
            t0 = time.perf_counter()
            motion_warp, _, _, _, data_mov_corrected = calFlow3d_Wei_v1.getMotion(
                ref_data_clip, move_data, option, verbose=False
            )

            # 如果内部用了 CuPy/GPU，建议同步后再停止计时
            cp.cuda.Stream.null.synchronize()
            warp_runtime_sec = time.perf_counter() - t0


            t0 = time.perf_counter()
            phase_map, motion_map, data_mov_mapped = calFlowCrossResolution.getMotion_v2(
                move_data, ref_data, option, verbose=False
            )

            cp.cuda.Stream.null.synchronize()
            map_runtime_sec = time.perf_counter() - t0
            
            ##############################################################################################
            ## baseline
            crop = 50
            z_ratio_eval = option['zRatio_HR']  # 或者你最终想用于物理误差的那个 zRatio

            ref_eval = simulation.crop_valid(ref_data_clip, crop=crop)
            mov_eval = simulation.crop_valid(move_data, crop=crop)
            gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)

            raw_intensity_mse = simulation.compute_intensity_mse(ref_eval, mov_eval)
            raw_intensity_rmse = simulation.compute_intensity_rmse(ref_eval, mov_eval)
            raw_intensity_ssim = simulation.compute_volume_ssim(ref_eval, mov_eval)


            ## warping method
            motion_warp_eval = simulation.crop_valid(motion_warp, crop=crop)
            data_mov_corrected_np = data_mov_corrected.get()
            warp_img_eval = simulation.crop_valid(data_mov_corrected_np, crop=crop)

            warp_intensity_mse = simulation.compute_intensity_mse(warp_img_eval, mov_eval)
            warp_intensity_rmse = simulation.compute_intensity_rmse(warp_img_eval, mov_eval)
            warp_intensity_ssim = simulation.compute_volume_ssim(warp_img_eval, mov_eval)

            warp_motion_metrics = simulation.compute_motion_metrics(
                motion_warp_eval,
                gt_motion_eval,
                z_ratio=z_ratio_eval,
                use_z=True
            )


            ## mapping method
            motion_map_eval = simulation.crop_valid(motion_map, crop=crop)
            map_img_eval = simulation.crop_valid(data_mov_mapped, crop=crop)

            map_intensity_mse = simulation.compute_intensity_mse(map_img_eval, mov_eval)
            map_intensity_rmse = simulation.compute_intensity_rmse(map_img_eval, mov_eval)
            map_intensity_ssim = simulation.compute_volume_ssim(map_img_eval, mov_eval)

            map_motion_metrics = simulation.compute_motion_metrics(
                motion_map_eval,
                gt_motion_eval,
                z_ratio=z_ratio_eval,
                use_z=True   # 如果你暂时只想比较xy，就改成 False
            )



            # print(f"[Intensity]  MSE raw:{np.mean((move_data - ref_data_clip)[50:-50,50:-50,:] ** 2):.4f}, MSE warp: {intensity_mse_warp:.4f}, MSE map: {intensity_mse_map:.4f}")
            # print(f"[Motion]   MSE raw:{np.mean(motion_data[50:-50,50:-50,:,:] ** 2):.4f}, MSE warp: {motion_mse_warp:.4f}, MSE map: {motion_mse_map:.4f}")

            print(
                f"[Intensity] "
                f"Raw MSE: {raw_intensity_mse:.4f}, RMSE: {raw_intensity_rmse:.4f}, SSIM: {raw_intensity_ssim:.4f} | "
                f"Warp MSE: {warp_intensity_mse:.4f}, RMSE: {warp_intensity_rmse:.4f}, SSIM: {warp_intensity_ssim:.4f} | "
                f"Map MSE: {map_intensity_mse:.4f}, RMSE: {map_intensity_rmse:.4f}, SSIM: {map_intensity_ssim:.4f}"
            )

            print(
                f"[Motion XY] "
                f"Warp EPE_xy: {warp_motion_metrics['motion_epe_xy_mean']:.4f}, "
                f"RMSE_xy: {warp_motion_metrics['motion_rmse_xy']:.4f} | "
                f"Map EPE_xy: {map_motion_metrics['motion_epe_xy_mean']:.4f}, "
                f"RMSE_xy: {map_motion_metrics['motion_rmse_xy']:.4f}"
            )

            if 'motion_epe_phys_mean' in warp_motion_metrics and 'motion_epe_phys_mean' in map_motion_metrics:
                print(
                    f"[Motion 3D/Phys] "
                    f"Warp EPE_phys: {warp_motion_metrics['motion_epe_phys_mean']:.4f}, "
                    f"RMSE_z: {warp_motion_metrics['motion_rmse_z']:.4f} | "
                    f"Map EPE_phys: {map_motion_metrics['motion_epe_phys_mean']:.4f}, "
                    f"RMSE_z: {map_motion_metrics['motion_rmse_z']:.4f}"
                )

            if exp_level not in results[group]:
                results[group][exp_level] = {
                    # baseline
                    'raw_intensity_mse': [],
                    'raw_intensity_rmse': [],
                    'raw_intensity_ssim': [],

                    # warp image metrics
                    'warp_intensity_mse': [],
                    'warp_intensity_rmse': [],
                    'warp_intensity_ssim': [],
                    'warp_runtime_sec': [],

                    # map image metrics
                    'map_intensity_mse': [],
                    'map_intensity_rmse': [],
                    'map_intensity_ssim': [],
                    'map_runtime_sec': [],

                    # warp motion metrics
                    'warp_motion_epe_xy_mean': [],
                    'warp_motion_epe_xy_median': [],
                    'warp_motion_rmse_xy': [],
                    'warp_motion_mae_y': [],
                    'warp_motion_mae_x': [],
                    'warp_motion_rmse_y': [],
                    'warp_motion_rmse_x': [],

                    # map motion metrics
                    'map_motion_epe_xy_mean': [],
                    'map_motion_epe_xy_median': [],
                    'map_motion_rmse_xy': [],
                    'map_motion_mae_y': [],
                    'map_motion_mae_x': [],
                    'map_motion_rmse_y': [],
                    'map_motion_rmse_x': [],
                }

                # 如果 use_z=True，会额外补这些 key
                if 'motion_epe_3d_mean' in warp_motion_metrics:
                    results[group][exp_level].update({
                        'warp_motion_epe_3d_mean': [],
                        'warp_motion_epe_3d_median': [],
                        'warp_motion_epe_phys_mean': [],
                        'warp_motion_epe_phys_median': [],
                        'warp_motion_rmse_z': [],
                        'warp_motion_mae_z': [],
                    })

                if 'motion_epe_3d_mean' in map_motion_metrics:
                    results[group][exp_level].update({
                        'map_motion_epe_3d_mean': [],
                        'map_motion_epe_3d_median': [],
                        'map_motion_epe_phys_mean': [],
                        'map_motion_epe_phys_median': [],
                        'map_motion_rmse_z': [],
                        'map_motion_mae_z': [],
                    })

            # -------------------------
            # append baseline metrics
            # -------------------------
            results[group][exp_level]['raw_intensity_mse'].append(raw_intensity_mse)
            results[group][exp_level]['raw_intensity_rmse'].append(raw_intensity_rmse)
            results[group][exp_level]['raw_intensity_ssim'].append(raw_intensity_ssim)

            # -------------------------
            # append warp image metrics
            # -------------------------
            results[group][exp_level]['warp_intensity_mse'].append(warp_intensity_mse)
            results[group][exp_level]['warp_intensity_rmse'].append(warp_intensity_rmse)
            results[group][exp_level]['warp_intensity_ssim'].append(warp_intensity_ssim)
            results[group][exp_level]['warp_runtime_sec'].append(warp_runtime_sec)

            # -------------------------
            # append map image metrics
            # -------------------------
            results[group][exp_level]['map_intensity_mse'].append(map_intensity_mse)
            results[group][exp_level]['map_intensity_rmse'].append(map_intensity_rmse)
            results[group][exp_level]['map_intensity_ssim'].append(map_intensity_ssim)
            results[group][exp_level]['map_runtime_sec'].append(map_runtime_sec)
            # -------------------------
            # append warp motion metrics
            # -------------------------
            results[group][exp_level]['warp_motion_epe_xy_mean'].append(warp_motion_metrics['motion_epe_xy_mean'])
            results[group][exp_level]['warp_motion_epe_xy_median'].append(warp_motion_metrics['motion_epe_xy_median'])
            results[group][exp_level]['warp_motion_rmse_xy'].append(warp_motion_metrics['motion_rmse_xy'])
            results[group][exp_level]['warp_motion_mae_y'].append(warp_motion_metrics['motion_mae_y'])
            results[group][exp_level]['warp_motion_mae_x'].append(warp_motion_metrics['motion_mae_x'])
            results[group][exp_level]['warp_motion_rmse_y'].append(warp_motion_metrics['motion_rmse_y'])
            results[group][exp_level]['warp_motion_rmse_x'].append(warp_motion_metrics['motion_rmse_x'])

            if 'motion_epe_3d_mean' in warp_motion_metrics:
                results[group][exp_level]['warp_motion_epe_3d_mean'].append(warp_motion_metrics['motion_epe_3d_mean'])
                results[group][exp_level]['warp_motion_epe_3d_median'].append(warp_motion_metrics['motion_epe_3d_median'])
                results[group][exp_level]['warp_motion_epe_phys_mean'].append(warp_motion_metrics['motion_epe_phys_mean'])
                results[group][exp_level]['warp_motion_epe_phys_median'].append(warp_motion_metrics['motion_epe_phys_median'])
                results[group][exp_level]['warp_motion_rmse_z'].append(warp_motion_metrics['motion_rmse_z'])
                results[group][exp_level]['warp_motion_mae_z'].append(warp_motion_metrics['motion_mae_z'])

            # -------------------------
            # append map motion metrics
            # -------------------------
            results[group][exp_level]['map_motion_epe_xy_mean'].append(map_motion_metrics['motion_epe_xy_mean'])
            results[group][exp_level]['map_motion_epe_xy_median'].append(map_motion_metrics['motion_epe_xy_median'])
            results[group][exp_level]['map_motion_rmse_xy'].append(map_motion_metrics['motion_rmse_xy'])
            results[group][exp_level]['map_motion_mae_y'].append(map_motion_metrics['motion_mae_y'])
            results[group][exp_level]['map_motion_mae_x'].append(map_motion_metrics['motion_mae_x'])
            results[group][exp_level]['map_motion_rmse_y'].append(map_motion_metrics['motion_rmse_y'])
            results[group][exp_level]['map_motion_rmse_x'].append(map_motion_metrics['motion_rmse_x'])

            if 'motion_epe_3d_mean' in map_motion_metrics:
                results[group][exp_level]['map_motion_epe_3d_mean'].append(map_motion_metrics['motion_epe_3d_mean'])
                results[group][exp_level]['map_motion_epe_3d_median'].append(map_motion_metrics['motion_epe_3d_median'])
                results[group][exp_level]['map_motion_epe_phys_mean'].append(map_motion_metrics['motion_epe_phys_mean'])
                results[group][exp_level]['map_motion_epe_phys_median'].append(map_motion_metrics['motion_epe_phys_median'])
                results[group][exp_level]['map_motion_rmse_z'].append(map_motion_metrics['motion_rmse_z'])
                results[group][exp_level]['map_motion_mae_z'].append(map_motion_metrics['motion_mae_z'])




=== Processing Repo rep01 (1/10) ===
  Processing art_R/1
[Intensity] Raw MSE: 3265.2476, RMSE: 57.1423, SSIM: 0.9524 | Warp MSE: 616.1254, RMSE: 24.8219, SSIM: 0.9786 | Map MSE: 465.2628, RMSE: 21.5700, SSIM: 0.9812
[Motion XY] Warp EPE_xy: 1.7311, RMSE_xy: 2.5399 | Map EPE_xy: 1.2230, RMSE_xy: 2.0791
[Motion 3D/Phys] Warp EPE_phys: 4.3353, RMSE_z: 1.4955 | Map EPE_phys: 3.0238, RMSE_z: 1.2407
  Processing art_R/2
[Intensity] Raw MSE: 3824.3408, RMSE: 61.8413, SSIM: 0.9515 | Warp MSE: 602.8695, RMSE: 24.5534, SSIM: 0.9790 | Map MSE: 338.9369, RMSE: 18.4102, SSIM: 0.9857
[Motion XY] Warp EPE_xy: 1.5945, RMSE_xy: 2.4253 | Map EPE_xy: 0.7645, RMSE_xy: 1.5647
[Motion 3D/Phys] Warp EPE_phys: 3.8850, RMSE_z: 1.3447 | Map EPE_phys: 2.1651, RMSE_z: 1.0350
  Processing art_R/3
[Intensity] Raw MSE: 3793.9346, RMSE: 61.5949, SSIM: 0.9520 | Warp MSE: 532.5383, RMSE: 23.0768, SSIM: 0.9861 | Map MSE: 280.7182, RMSE: 16.7546, SSIM: 0.9916
[Motion XY] Warp EPE_xy: 1.1866, RMSE_xy: 2.0529 | Map EPE_x

save_result

In [3]:
output_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/wbi.npy"
processed_results = {}

for group in experiment_groups:
    if group not in results or not results[group]:
        continue

    sorted_exp_levels = sorted(results[group].keys(), key=int)

    processed_results[group] = {
        'labels': sorted_exp_levels
    }

    # 收集这个 group 下所有出现过的 metric 名称
    metric_names = set()
    for lv in sorted_exp_levels:
        metric_names.update(results[group][lv].keys())

    # 对每个 metric 自动计算平均值和标准差
    for metric_name in sorted(metric_names):
        processed_results[group][f'avg_{metric_name}'] = [
            float(np.nanmean(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]
        processed_results[group][f'std_{metric_name}'] = [
            float(np.nanstd(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]

np.save(output_file, {
    'raw_results': results,
    'processed_results': processed_results
}, allow_pickle=True)

print(f"Saved all results to {output_file}")

Saved all results to /home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/wbi.npy


In [6]:
processed_results[group]

{'labels': ['1', '2', '3', '4', '5', '6', '7', '8', '9'],
 'avg_map_intensity_mse': [215.95549248301296,
  228.5016389253728,
  270.38517543892374,
  304.5397685150786,
  398.9397489579207,
  475.7335546207116,
  550.1260436930256,
  681.4334553155293,
  860.6750809356014],
 'std_map_intensity_mse': [51.42304134561526,
  43.20416767351873,
  45.37089123244798,
  46.418822976555845,
  59.196026215943164,
  44.32682155162133,
  44.552042915624725,
  60.62894340770598,
  51.85234798623447],
 'avg_map_intensity_rmse': [14.599078165320554,
  15.051261353086236,
  16.377417136706857,
  17.40062899210968,
  19.918191590269117,
  21.787119832371772,
  23.435872174600142,
  26.078928719155595,
  29.32412800302244],
 'std_map_intensity_rmse': [1.6800027398411193,
  1.4004180113328557,
  1.4715241653552327,
  1.3258503663802383,
  1.4850564740956465,
  1.0271144194218236,
  0.9412433844805618,
  1.150187887577731,
  0.8778369996512676],
 'avg_map_intensity_ssim': [0.9931160272772754,
  0.99176852